In [25]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = '../data/census_acs/car_ownership.csv'
df_raw = pd.read_csv(file_path)

# 2. Inspect the raw state (showing the first 2 rows to highlight the metadata row)
print("--- 1. RAW DATA INSPECTION ---")
print(df_raw[['GEO_ID', 'B08141_001E', 'B08141_002E']].head(2))

print("\nRaw Data Types:")
print(df_raw[['GEO_ID', 'B08141_001E', 'B08141_002E']].dtypes)


--- 1. RAW DATA INSPECTION ---
                 GEO_ID       B08141_001E  \
0             Geography  Estimate!!Total:   
1  1400000US36005000100                 0   

                              B08141_002E  
0  Estimate!!Total:!!No vehicle available  
1                                       0  

Raw Data Types:
GEO_ID         object
B08141_001E    object
B08141_002E    object
dtype: object


In [ ]:
# 3. Clean and Transform
print("\n--- 2. APPLYING FIXES & TRANSFORMATIONS ---")
# Drop the first row (metadata string descriptions)
df_clean = df_raw.drop(index=0).copy()

# Rename the essential columns based on our mapping
column_mapping = {
    'GEO_ID': 'GEOID',
    'B08141_001E': 'total_households',
    'B08141_002E': 'zero_vehicle_households'
}
df_clean = df_clean.rename(columns=column_mapping)

# Keep only the essential columns (Discarding Margin of Error and irrelevant demographics)
df_clean = df_clean[['GEOID', 'total_households', 'zero_vehicle_households']]

# Convert household counts from strings to numeric types
df_clean['total_households'] = pd.to_numeric(df_clean['total_households'], errors='coerce')
df_clean['zero_vehicle_households'] = pd.to_numeric(df_clean['zero_vehicle_households'], errors='coerce')

# Drop missing values and tracts with zero total households (to prevent division by zero)
initial_rows = len(df_clean)
df_clean = df_clean.dropna(subset=['total_households', 'zero_vehicle_households'])
df_clean = df_clean[df_clean['total_households'] > 0]
dropped_rows = initial_rows - len(df_clean)
print(f"Dropped {dropped_rows} tracts due to nulls or zero total households.")

# Feature Engineering: Calculate the car ownership rate
df_clean['car_ownership_rate'] = df_clean['zero_vehicle_households'] / df_clean['total_households']

# 4. Final Validation & Inspection
print("\n--- 3. CLEANED DATA INSPECTION (VALIDATION) ---")
print(df_clean.head(5))
print("\nCleaned Data Types:")
print(df_clean.dtypes)

# Validate logical constraints
assert df_clean.isnull().sum().sum() == 0, "Null values still exist!"
assert (df_clean['total_households'] == 0).sum() == 0, "Zero household tracts remain!"
assert df_clean['car_ownership_rate'].between(0, 1).all(), "Car ownership rate out of bounds!"

# Save to a new CSV for downstream processes
output_path = 'car_ownership_cleaned.csv'
df_clean.to_csv(output_path, index=False)


--- 2. APPLYING FIXES & TRANSFORMATIONS ---
Dropped 99 tracts due to nulls or zero total households.

--- 3. CLEANED DATA INSPECTION (VALIDATION) ---
                  GEOID  total_households  zero_vehicle_households  \
2  1400000US36005000200              2431                      616   
3  1400000US36005000400              3043                      251   
4  1400000US36005001600              2465                      620   
5  1400000US36005001901              1232                      866   
6  1400000US36005001902               825                      249   

   car_ownership_rate  
2            0.253394  
3            0.082484  
4            0.251521  
5            0.702922  
6            0.301818  

Cleaned Data Types:
GEOID                       object
total_households             int64
zero_vehicle_households      int64
car_ownership_rate         float64
dtype: object


<>:43: SyntaxWarning: invalid escape sequence '\A'
<>:43: SyntaxWarning: invalid escape sequence '\A'
C:\Users\ajakd\AppData\Local\Temp\ipykernel_30340\1789127249.py:43: SyntaxWarning: invalid escape sequence '\A'
  output_path = 'C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\data\processed'
C:\Users\ajakd\AppData\Local\Temp\ipykernel_30340\1789127249.py:43: SyntaxWarning: invalid escape sequence '\A'
  output_path = 'C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\data\processed'


OSError: Cannot save file into a non-existent directory: 'C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progressyp\data'

(US Census Bureau, 2023)